# Binary Cross-Entropy (BCE / Log Loss) — Beginner Masterclass

**Reading:** [GeeksforGeeks — Binary Cross Entropy / Log Loss](https://www.geeksforgeeks.org/deep-learning/binary-cross-entropy-log-loss-for-binary-classification/) and [Loss Functions in Deep Learning](https://www.geeksforgeeks.org/deep-learning/loss-functions-in-deep-learning/)

Learn BCE with worked arithmetic, plain Python, NumPy, PyTorch, TensorFlow/Keras, plots, and a binary-classification training experiment.

## Learning goals
- Understand binary labels, probabilities, and log loss.
- Calculate BCE by hand.
- Understand why confident wrong predictions receive a large penalty.
- Know the difference between probabilities and logits.
- Use `BCELoss`, `BCEWithLogitsLoss`, and Keras `BinaryCrossentropy`.
- Avoid double sigmoid, shape mismatches, and invalid target values.

## 1. Formula and intuition

Binary Cross-Entropy is common for classification with two mutually exclusive classes, encoded 0 and 1.

\[
\boxed{
L_{\mathrm{BCE}}=-\frac{1}{N}\sum_{i=1}^{N}
\left[y_i\log(p_i)+(1-y_i)\log(1-p_i)\right]
}
\]

Here \(y_i\in\{0,1\}\), \(p_i\in[0,1]\) is the predicted probability for class 1, and logs are natural logarithms.

For one sample:
- If \(y=1\), loss \(=-\log(p)\).
- If \(y=0\), loss \(=-\log(1-p)\).

Loss is low when the probability assigned to the true label is high. It is high when the model is confidently wrong.

## 2. Hand-worked example

True labels: \(y=[1,0,1,1]\)

Predicted probabilities for class 1: \(p=[0.9,0.2,0.7,0.4]\)

Individual losses:
- \(y=1,p=0.9:\;-\log(0.9)\)
- \(y=0,p=0.2:\;-\log(1-0.2)=-\log(0.8)\)
- \(y=1,p=0.7:\;-\log(0.7)\)
- \(y=1,p=0.4:\;-\log(0.4)\)

The batch BCE is the mean of these individual losses.

In [ ]:
# Pure Python only
import math
y_true = [1, 0, 1, 1]
y_prob = [0.9, 0.2, 0.7, 0.4]
EPS = 1e-7
individual = []
for y, p in zip(y_true, y_prob):
    # Clip for this manual demonstration so log(0) is avoided.
    p = min(max(p, EPS), 1 - EPS)
    item = -(y * math.log(p) + (1-y) * math.log(1-p))
    individual.append(item)
bce_manual = sum(individual) / len(individual)
print("Labels:", y_true)
print("Probabilities:", y_prob)
print("Per-example losses:", [round(v, 6) for v in individual])
print("Sum:", sum(individual))
print("Mean BCE:", bce_manual)

## 3. NumPy implementation

In [ ]:
import numpy as np
y_np = np.array(y_true, dtype=np.float64)
p_np = np.array(y_prob, dtype=np.float64)
p_safe = np.clip(p_np, 1e-7, 1 - 1e-7)
bce_np = -np.mean(y_np*np.log(p_safe) + (1-y_np)*np.log(1-p_safe))
print("NumPy BCE:", bce_np)
assert np.isclose(bce_np, bce_manual)

## 4. Visualize the penalty

When the true label is 1, the loss \(-\log(p)\) explodes as \(p\) approaches zero. When the true label is 0, \(-\log(1-p)\) grows as \(p\) approaches one.

In [ ]:
import matplotlib.pyplot as plt
p = np.linspace(0.001, 0.999, 500)
plt.figure(figsize=(8,4.5))
plt.plot(p, -np.log(p), label="True label = 1")
plt.plot(p, -np.log(1-p), label="True label = 0")
plt.xlabel("Predicted probability for class 1")
plt.ylabel("BCE for one sample")
plt.title("Binary cross-entropy penalizes confident mistakes")
plt.ylim(0, 7)
plt.legend()
plt.grid(alpha=.25)
plt.show()

## 5. PyTorch: `BCELoss` vs `BCEWithLogitsLoss`

A **logit** is a raw, unbounded model score \(z\). Sigmoid maps logits to probabilities:

\[
p=\sigma(z)=\frac{1}{1+e^{-z}}.
\]

- `nn.BCELoss`: takes probabilities in \([0,1]\).
- `nn.BCEWithLogitsLoss`: takes raw logits, combines sigmoid and BCE in a more numerically stable operation, and is generally preferred for training.

**Do not apply sigmoid twice.** If you use `BCEWithLogitsLoss`, pass the model's raw logits.

In [ ]:
import torch
import torch.nn as nn

yt = torch.tensor(y_true, dtype=torch.float32)
p_t = torch.tensor(y_prob, dtype=torch.float32)
loss_prob = nn.BCELoss()(p_t, yt)

# Convert probabilities to their corresponding logits for an equivalence demo.
logits_equivalent = torch.log(p_t / (1 - p_t))
loss_logits = nn.BCEWithLogitsLoss()(logits_equivalent, yt)

print("PyTorch BCELoss (probabilities):", loss_prob.item())
print("PyTorch BCEWithLogitsLoss (logits):", loss_logits.item())
assert np.isclose(loss_prob.item(), loss_logits.item(), atol=1e-6)

## 6. TensorFlow / Keras BCE

Match `from_logits` to the model output:
- Sigmoid output probabilities → `BinaryCrossentropy(from_logits=False)`.
- Raw output logits → `BinaryCrossentropy(from_logits=True)`.

The logits path is usually preferred for stable training.

In [ ]:
import tensorflow as tf
yt_tf = tf.constant(y_true, dtype=tf.float32)
p_tf = tf.constant(y_prob, dtype=tf.float32)
logits_tf = tf.math.log(p_tf / (1-p_tf))

keras_prob_loss = tf.keras.losses.BinaryCrossentropy(from_logits=False)(yt_tf, p_tf)
keras_logit_loss = tf.keras.losses.BinaryCrossentropy(from_logits=True)(yt_tf, logits_tf)
print("Keras BCE on probabilities:", float(keras_prob_loss.numpy()))
print("Keras BCE on logits:", float(keras_logit_loss.numpy()))
assert np.isclose(float(keras_prob_loss.numpy()), bce_manual, atol=1e-5)

## 7. Train a binary classifier with BCEWithLogitsLoss

We generate 2D synthetic data with a probabilistic binary label. A linear model produces one logit per sample; `BCEWithLogitsLoss` handles the sigmoid operation internally. No dataset upload is needed.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
SEED = 42
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

X = rng.normal(size=(1200,2)).astype(np.float32)
true_logits = 1.8*X[:,0] - 1.2*X[:,1] + 0.25
true_probs = 1/(1+np.exp(-true_logits))
y = rng.binomial(1, true_probs).astype(np.float32)

ids = rng.permutation(len(X))
train_ids, test_ids = ids[:900], ids[900:]
X_train, X_test = X[train_ids], X[test_ids]
y_train, y_test = y[train_ids], y[test_ids]

loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train).reshape(-1,1)),
    batch_size=32, shuffle=True
)
X_test_t = torch.from_numpy(X_test)
y_test_t = torch.from_numpy(y_test).reshape(-1,1)

model = nn.Linear(2,1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
criterion = nn.BCEWithLogitsLoss()
train_losses, test_losses, test_accs = [], [], []

for epoch in range(60):
    model.train()
    total, count = 0., 0
    for xb, yb in loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total += loss.item()*len(xb); count += len(xb)
    train_losses.append(total/count)

    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_t)
        test_losses.append(criterion(test_logits, y_test_t).item())
        pred_labels = (torch.sigmoid(test_logits) >= .5).float()
        test_accs.append((pred_labels == y_test_t).float().mean().item())

print("Final test BCE:", test_losses[-1])
print("Final test accuracy:", round(test_accs[-1]*100, 2), "%")

In [ ]:
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(train_losses, label="Train BCE")
plt.plot(test_losses, label="Test BCE")
plt.xlabel("Epoch"); plt.ylabel("BCE"); plt.title("Training loss")
plt.legend(); plt.grid(alpha=.25)
plt.subplot(1,2,2)
plt.plot(np.array(test_accs)*100)
plt.xlabel("Epoch"); plt.ylabel("Test accuracy (%)"); plt.title("Test accuracy")
plt.grid(alpha=.25)
plt.tight_layout(); plt.show()

## 8. Common mistakes and debugging

- **Raw logits passed to `BCELoss`:** use `BCEWithLogitsLoss` for logits.
- **Sigmoid twice:** do not apply sigmoid and then use `BCEWithLogitsLoss`.
- **Shape mismatch:** if logits are `(N,1)`, targets should ordinarily be `(N,1)` too.
- **Wrong dtype:** PyTorch BCE targets should normally be floating-point 0/1 values.
- **Out-of-range predictions:** `BCELoss` expects probabilities in \([0,1]\).
- **`log(0)` in manual code:** clip probabilities or use stable logits-based library loss.
- **Label encoding:** BCE normally expects binary labels 0 and 1.

## 9. Quick quiz

1. Which PyTorch loss expects raw logits? **BCEWithLogitsLoss.**
2. Why penalize confident mistakes? The negative logarithm becomes large when the true class probability is near zero.
3. Should sigmoid outputs be passed into BCEWithLogitsLoss? **No.** Use raw logits.
4. Which Keras setting takes raw logits? `from_logits=True`.

## Sources and official documentation

### All six reading links supplied for this series
- [GeeksforGeeks — Loss Functions in Deep Learning](https://www.geeksforgeeks.org/deep-learning/loss-functions-in-deep-learning/)
- [GeeksforGeeks — Mean Squared Error](https://www.geeksforgeeks.org/maths/mean-squared-error/)
- [GeeksforGeeks — MAE in PyTorch](https://www.geeksforgeeks.org/python/how-to-measure-the-mean-absolute-error-mae-in-pytorch/)
- [Medium — Mean Absolute Error](https://medium.com/@20__80__/mean-absolute-error-mae-machine-learning-ml-b9b4afc63077)
- [GeeksforGeeks — Binary Cross-Entropy](https://www.geeksforgeeks.org/deep-learning/binary-cross-entropy-log-loss-for-binary-classification/)
- [GeeksforGeeks — Categorical Cross-Entropy](https://www.geeksforgeeks.org/deep-learning/categorical-cross-entropy-in-multi-class-classification/)

### API documentation for implementation/debugging
- [PyTorch BCELoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCELoss.html)
- [PyTorch BCEWithLogitsLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)
- [Keras probabilistic losses](https://keras.io/api/losses/probabilistic_losses/)
- [NumPy log](https://numpy.org/doc/stable/reference/generated/numpy.log.html)

The notebook summarizes source material rather than reproducing the complete articles verbatim. When debugging, inspect tensor shape, dtype, value range, and whether the model outputs logits or probabilities.